# Solar UV and X-ray Spectral Diagnostics — Implementation
# 태양 UV·X선 분광 진단 구현

**Paper**: Del Zanna, G., & Mason, H. E. (2018). *Solar UV and X-ray spectral diagnostics*. Living Reviews in Solar Physics, 15:5.

## Goals / 목표
1. Build simplified contribution functions $G(T)$ for representative ions (Fe IX, Fe XII, Fe XXIV).
2. Reproduce density-sensitive line-ratio S-curve (Fe XII 195.12/186.89 analogue).
3. Build temperature diagnostic from a two-line ratio.
4. Perform a simple regularized DEM inversion from synthetic line intensities.
5. Plot ionization equilibrium fractions vs T for iron.
6. Demonstrate isothermal EM-loci diagnostic.

1. 대표 이온(Fe IX, Fe XII, Fe XXIV)에 대한 간략화된 기여함수 $G(T)$ 구축
2. 밀도 민감 line ratio S-곡선(Fe XII 195.12/186.89 유사) 재현
3. 두 선의 비로부터 온도 진단
4. 합성 라인 강도로부터 간단한 정칙화 DEM 역산 수행
5. 철의 이온화 평형 분율 vs T 그래프 작성
6. 등온 EM-loci 진단 시연

Note: This is a pedagogical reproduction. Production analysis should use the CHIANTI atomic database (via `ChiantiPy` in Python or `SolarSoft` CHIANTI in IDL).

In [ ]:
"""Imports and constants used across the notebook.

All quantities in CGS unless noted.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve

# Physical constants (CGS)
K_B = 1.380649e-16        # erg/K
H_PLANCK = 6.62607015e-27 # erg s
C_LIGHT = 2.998e10        # cm/s
M_E = 9.1093837e-28       # g
EV_TO_ERG = 1.602e-12

np.random.seed(42)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## 1. Contribution function $G(T)$ for representative ions / 대표 이온의 기여함수

$G(T,N_e)$ encodes all atomic physics for a line. For a *strong allowed* line from the ground state, $G$ is often approximated by a Gaussian in $\log T$ centered at the ion's peak-abundance temperature with FWHM $\Delta\log T \approx 0.2$–$0.3$.

$G(T,N_e)$는 선 하나의 원자물리를 전부 담는 함수이다. 기저 준위에서 여기되는 강한 허용 선의 경우 $\log T$에서 중심이 이온의 peak abundance 온도이고 FWHM $\Delta\log T \approx 0.2$–$0.3$인 Gaussian으로 근사된다.

Reference peak temperatures (CHIANTI v8): Fe IX $\log T=5.85$, Fe XII $\log T=6.20$, Fe XXIV $\log T=7.15$.

In [ ]:
def contribution_function(log_T, log_T_peak, width=0.15, amplitude=1.0):
    """Gaussian approximation of G(T) as function of log10(T).

    Args:
        log_T: log10(temperature/K), numpy array.
        log_T_peak: log10 of the ion's peak abundance temperature.
        width: standard deviation in log T (default 0.15 -> FWHM~0.35).
        amplitude: peak value of G in units of 1e-25 erg cm^3 s^-1.

    Returns:
        G(T) as numpy array (same shape as log_T), in units of 1e-25 erg cm^3 s^-1.
    """
    return amplitude * np.exp(-0.5 * ((log_T - log_T_peak) / width) ** 2)

log_T = np.linspace(5.0, 7.5, 300)
ions = {
    'Fe IX (171 A)': (5.85, 3.0, 'tab:blue'),
    'Fe XII (195.12 A)': (6.20, 2.5, 'tab:green'),
    'Fe XXIV (192 A)': (7.15, 1.5, 'tab:red'),
}

plt.figure(figsize=(7, 4.5))
for name, (logT_peak, amp, color) in ions.items():
    G = contribution_function(log_T, logT_peak, width=0.18, amplitude=amp)
    plt.plot(log_T, G, label=name, color=color, lw=2)
plt.xlabel('log$_{10}$ T [K]')
plt.ylabel('G(T) [arb. units, 10$^{-25}$ erg cm$^3$ s$^{-1}$]')
plt.title('Contribution functions: Fe IX, Fe XII, Fe XXIV')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Peak temperatures:')
print('  Fe IX: 10^5.85 =', f'{10**5.85:.2e}', 'K (~ 0.7 MK)')
print('  Fe XII: 10^6.20 =', f'{10**6.20:.2e}', 'K (~ 1.6 MK)')
print('  Fe XXIV: 10^7.15 =', f'{10**7.15:.2e}', 'K (~ 14 MK, flare)')

## 2. Density-sensitive line ratio (Fe XII 186.89/195.12 analogue) / 밀도 민감 선비

Two-level model: metastable population ratio depends on balance between collisional de-excitation and spontaneous decay.

$$\frac{N_m}{N_g} = \frac{N_e C^e_{g,m}}{N_e C^e_{m,g} + A_{m,g}}.$$

Ratio of forbidden (metastable-fed) to allowed (ground-fed) line gives the classic S-curve:
- $N_e \to 0$: ratio $\propto N_e$ (metastable populated collisionally against A)
- $N_e \to \infty$: ratio saturates (Boltzmann)

두 준위 모델: metastable 준위의 population은 충돌 탈여기와 자발방출의 균형으로 결정된다. 저밀도에서 비 ∝ N_e, 고밀도에서는 포화.

In [ ]:
def density_sensitive_ratio(log_Ne, A_mg=60.0, Ce_mg=3e-8, ratio_low=0.1, ratio_high=1.4):
    """S-curve ratio of forbidden/allowed line as function of log10(N_e).

    This mimics the Fe XII 186.89/195.12 A emissivity ratio shape.

    Args:
        log_Ne: log10(electron density / cm^-3).
        A_mg: spontaneous decay rate of metastable level (s^-1).
        Ce_mg: collisional de-excitation rate coefficient (cm^3/s).
        ratio_low: asymptotic ratio at very low density.
        ratio_high: asymptotic ratio at very high density.

    Returns:
        Ratio value for each log_Ne.
    """
    Ne = 10.0 ** log_Ne
    x = Ne * Ce_mg / A_mg
    return ratio_low + (ratio_high - ratio_low) * x / (1.0 + x)

log_Ne_arr = np.linspace(7, 12, 200)
ratios = density_sensitive_ratio(log_Ne_arr)

plt.figure(figsize=(7, 4.5))
plt.plot(log_Ne_arr, ratios, 'b-', lw=2)
plt.axhline(0.1, color='gray', ls=':', alpha=0.5)
plt.axhline(1.4, color='gray', ls=':', alpha=0.5)
plt.xlabel('log$_{10}$ N$_e$ [cm$^{-3}$]')
plt.ylabel('Line ratio (186.89 / 195.12)')
plt.title('Density-sensitive line-ratio S-curve (Fe XII analogue)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Invert: given an observed ratio, find density
observed_ratio = 0.5
idx = np.argmin(np.abs(ratios - observed_ratio))
print(f'Observed ratio = {observed_ratio}')
print(f'Implied log N_e ~ {log_Ne_arr[idx]:.2f} (N_e ~ {10**log_Ne_arr[idx]:.2e} cm^-3)')

## 3. Temperature diagnostic from a two-line ratio / 두 선 비로부터 온도 진단

For two allowed lines from ground with different excitation energies $\Delta E_j$, $\Delta E_k$ (from Eq. 105 in the review):
$$\frac{I_{g,j}}{I_{g,k}} = \frac{\Delta E_{g,j}\,\Upsilon_{g,j}}{\Delta E_{g,k}\,\Upsilon_{g,k}}\exp\!\left[\frac{\Delta E_{g,k}-\Delta E_{g,j}}{k_B T}\right].$$

Sensitivity requires $(\Delta E_k-\Delta E_j)/k_BT \gtrsim 1$.

같은 이온 내 두 허용선의 강도비는 지수적으로 T에 의존한다. (ΔE_k−ΔE_j)/k_BT >> 1 조건이 민감도의 핵심.

In [ ]:
def temperature_ratio(T, dE_j_eV=50.0, dE_k_eV=90.0, Y_j=2.0, Y_k=3.0):
    """Two-line temperature-sensitive ratio I_{g,j}/I_{g,k}.

    Args:
        T: temperature (K), numpy array.
        dE_j_eV, dE_k_eV: excitation energies in eV for levels j, k.
        Y_j, Y_k: thermally-averaged collision strengths.

    Returns:
        Ratio value for each T.
    """
    dE_j = dE_j_eV * EV_TO_ERG
    dE_k = dE_k_eV * EV_TO_ERG
    prefactor = (dE_j * Y_j) / (dE_k * Y_k)
    return prefactor * np.exp((dE_k - dE_j) / (K_B * T))

T_arr = np.logspace(5.5, 7.0, 200)
ratio = temperature_ratio(T_arr)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogx(T_arr, ratio, 'g-', lw=2)
ax.set_xlabel('T [K]')
ax.set_ylabel('Line ratio I$_j$/I$_k$')
ax.set_title('Temperature-sensitive line ratio ($\\Delta$E$_j$=50 eV, $\\Delta$E$_k$=90 eV)')
ax.set_yscale('log')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

observed_ratio = 0.5
idx = np.argmin(np.abs(np.log10(ratio) - np.log10(observed_ratio)))
print(f'Observed ratio = {observed_ratio}')
print(f'Implied T ~ {T_arr[idx]:.2e} K (log T = {np.log10(T_arr[idx]):.2f})')

## 4. Simple regularized DEM inversion / 간단한 정칙화 DEM 역산

Given observed line intensities $I_i = \int C_i(T)\,\mathrm{DEM}(T)\, dT$, recover $\mathrm{DEM}(T)$.

Discretize $T$ on a grid, then $I_i = \sum_j C_{ij}\,\mathrm{DEM}_j\,\Delta T_j$ → matrix form $\mathbf{I} = \mathbf{K}\mathbf{d}$. Since this is underdetermined, we apply Tikhonov regularization:
$$\hat{\mathbf{d}} = \arg\min \|\mathbf{K}\mathbf{d}-\mathbf{I}\|^2 + \lambda^2\|\mathbf{L}\mathbf{d}\|^2,$$
with $\mathbf{L}$ a smoothness operator (second derivative).

관측된 여러 선 강도 $I_i$에서 DEM(T)를 복원하는 문제는 제1종 Fredholm 적분으로 ill-posed이다. Tikhonov 정칙화로 2차 미분을 penalty로 주어 매끈러운 해를 얻는다.

In [ ]:
def contribution_function_2(log_T, log_T_peak, width=0.15, amplitude=1.0):
    """Same as section 1 contribution_function, replicated here for clarity."""
    return amplitude * np.exp(-0.5 * ((log_T - log_T_peak) / width) ** 2)

logT_grid = np.linspace(5.5, 7.0, 40)
T_grid = 10 ** logT_grid
dlogT = logT_grid[1] - logT_grid[0]

def synthetic_dem(logT):
    """Ground-truth DEM with two peaks (1 MK warm loops + 3 MK AR core)."""
    g1 = np.exp(-0.5 * ((logT - 6.00) / 0.12) ** 2)
    g2 = np.exp(-0.5 * ((logT - 6.50) / 0.08) ** 2)
    return 1e22 * (0.8 * g1 + 1.5 * g2)  # cm^-5 K^-1

dem_true = synthetic_dem(logT_grid)

# 10 synthetic lines with peaks spanning log T = 5.6 to 6.9
logT_peaks_lines = np.linspace(5.6, 6.9, 10)
line_names = [f'Line {i+1} (logT_peak={p:.2f})' for i, p in enumerate(logT_peaks_lines)]

# Kernel matrix: integrate C(T)*DEM(T) dT using log-T grid -> dT = T*ln10*dlogT
K = np.zeros((len(logT_peaks_lines), len(logT_grid)))
for i, lp in enumerate(logT_peaks_lines):
    C = contribution_function_2(logT_grid, lp, width=0.15, amplitude=1.0)
    K[i, :] = C * T_grid * np.log(10) * dlogT

# Synthetic observed intensities with 5% Gaussian noise
I_obs_clean = K @ dem_true
noise_frac = 0.05
I_obs = I_obs_clean * (1 + noise_frac * np.random.randn(len(I_obs_clean)))
sigma_I = noise_frac * I_obs_clean

# Second-difference regularization operator
n = len(logT_grid)
L = np.diag(-2 * np.ones(n)) + np.diag(np.ones(n - 1), 1) + np.diag(np.ones(n - 1), -1)
L = L[1:-1, :]

def tikhonov_solve(K, I, L, lam):
    """Solve regularized least squares using normal equations."""
    A = K.T @ K + (lam ** 2) * (L.T @ L)
    b = K.T @ I
    return solve(A, b)

lam = 1e21
dem_rec = tikhonov_solve(K, I_obs, L, lam)
dem_rec = np.clip(dem_rec, 0, None)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(logT_grid, dem_true, 'k-', lw=2, label='True DEM')
ax.plot(logT_grid, dem_rec, 'r--', lw=2, label=f'Tikhonov recovered ($\\lambda$={lam:.0e})')
ax.set_xlabel('log$_{10}$ T [K]')
ax.set_ylabel('DEM(T) [cm$^{-5}$ K$^{-1}$]')
ax.set_title('Regularized DEM inversion from 10 synthetic lines')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

I_rec = K @ dem_rec
print('Intensity residuals (obs - reconstructed) / sigma:')
for i, (io, ir, s) in enumerate(zip(I_obs, I_rec, sigma_I)):
    print(f'  {line_names[i]}: (I_obs - I_rec)/sigma = {(io - ir) / s:+.2f}')

## 5. Iron ionization equilibrium vs T / 철 이온화 평형

In equilibrium, $\dot N_r = 0$ gives the fractional abundance $N(Z^{+r})/N(Z)$ as function of T. The paper uses CHIANTI's ionization-equilibrium tables; we approximate each ion's fraction as a Gaussian centered at its peak-abundance temperature.

이온화 평형에서 $\dot N_r = 0$으로부터 철 각 이온의 분율 $N(\text{Fe}^{+r})/N(\text{Fe})$을 얻는다. 여기서는 각 이온을 peak T에 중심을 둔 Gaussian으로 근사한다.

In [ ]:
# Peak abundance log T for iron ions (approximate, based on CHIANTI)
iron_peaks = {
    'Fe VIII': 5.70, 'Fe IX': 5.85, 'Fe X': 6.00, 'Fe XI': 6.10,
    'Fe XII': 6.20, 'Fe XIII': 6.25, 'Fe XIV': 6.30, 'Fe XV': 6.35,
    'Fe XVI': 6.45, 'Fe XVII': 6.75, 'Fe XVIII': 6.85, 'Fe XIX': 6.95,
    'Fe XX': 7.00, 'Fe XXIII': 7.15, 'Fe XXIV': 7.20, 'Fe XXV': 7.50,
}

logT = np.linspace(5.3, 8.0, 400)
plt.figure(figsize=(9, 5))
cmap = plt.cm.plasma
fractions = np.zeros((len(iron_peaks), len(logT)))
for i, (name, lp) in enumerate(iron_peaks.items()):
    fractions[i, :] = np.exp(-0.5 * ((logT - lp) / 0.12) ** 2)

col_sum = fractions.sum(axis=0)
fractions_norm = fractions / col_sum[None, :]

for i, (name, lp) in enumerate(iron_peaks.items()):
    plt.plot(logT, fractions_norm[i, :], label=name, color=cmap(i / len(iron_peaks)))
plt.xlabel('log$_{10}$ T [K]')
plt.ylabel('Fractional abundance N(Fe$^{+r}$)/N(Fe)')
plt.title('Iron ionization equilibrium (approximate)')
plt.legend(ncol=3, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.15))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

query_logT = 6.3
idx = np.argmin(np.abs(logT - query_logT))
top = np.argsort(fractions_norm[:, idx])[::-1][:3]
names = list(iron_peaks.keys())
print(f'At log T = {query_logT} (T = {10**query_logT:.2e} K), top ions:')
for i in top:
    print(f'  {names[i]}: fraction = {fractions_norm[i, idx]:.3f}')

## 6. Isothermal EM-loci diagnostic / 등온 EM-loci 진단

For each observed line, $EM_L(T) = I_{obs}/G(T)$ is an upper bound on the emission measure. If the plasma is isothermal at some $T_0$, all EM-loci curves cross at $T_0$. If the plasma is multi-thermal, curves envelop the DEM.

각 관측 선에 대해 $EM_L(T) = I_{obs}/G(T)$는 EM 상한 곡선이다. 플라스마가 $T_0$에서 등온이면 모든 EM-loci 곡선이 $T_0$에서 교차한다.

In [ ]:
# Assume an isothermal plasma at T=3 MK with EM = 1e28 cm^-5 (AR core)
T_iso = 3e6
logT_iso = np.log10(T_iso)
EM_iso = 1e28

line_peaks = [5.9, 6.1, 6.3, 6.45, 6.65]
line_labels = ['Fe IX (171)', 'Fe XI (180)', 'Fe XIV (274)', 'Fe XVI (335)', 'Fe XVII (204)']

logT_arr = np.linspace(5.6, 7.0, 300)

I_list = []
for lp in line_peaks:
    G_atT = contribution_function_2(np.array([logT_iso]), lp, width=0.15, amplitude=1e-25)[0]
    I_list.append(G_atT * EM_iso)

plt.figure(figsize=(7.5, 5))
for lp, lbl, I_obs_val in zip(line_peaks, line_labels, I_list):
    G_curve = contribution_function_2(logT_arr, lp, width=0.15, amplitude=1e-25)
    G_curve_safe = np.where(G_curve > 1e-30, G_curve, 1e-30)
    EM_L = I_obs_val / G_curve_safe
    plt.plot(logT_arr, np.log10(EM_L), label=lbl, lw=2)

plt.axvline(logT_iso, color='k', ls=':', alpha=0.5, label=f'True T = {T_iso:.0e} K')
plt.axhline(np.log10(EM_iso), color='k', ls=':', alpha=0.5)
plt.xlabel('log$_{10}$ T [K]')
plt.ylabel('log$_{10}$ EM$_L$ [cm$^{-5}$]')
plt.title('EM-loci diagnostic for isothermal plasma (T=3 MK, log EM=28)')
plt.ylim(25, 31)
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'All EM-loci curves should meet near log T = {logT_iso:.2f}, log EM = {np.log10(EM_iso):.2f}')

## 7. Summary / 요약

This notebook has reproduced, in simplified pedagogical form, six of the core diagnostics developed in Del Zanna & Mason (2018):

1. **Contribution function $G(T)$**: sharp Gaussian peaks at ion's peak-abundance $T$.
2. **Density-sensitive line ratio**: S-curve from metastable-level population balance; the transition log $N_e$ is controlled by $A_{m,g}/C^e_{m,g}$.
3. **Temperature diagnostic**: exponential sensitivity $\exp((\Delta E_k-\Delta E_j)/k_BT)$.
4. **Regularized DEM inversion**: Tikhonov second-derivative penalty recovers a smooth DEM from a handful of intensities.
5. **Iron ionization equilibrium**: fractional abundances peaked at successive log $T$ values — gives the temperature axis for interpreting each iron line.
6. **EM-loci isothermal diagnostic**: crossing of $I/G(T)$ curves signals a near-isothermal plasma.

이 노트북은 리뷰의 핵심 진단 6가지를 간략화해 재현했다. 실제 태양 관측 해석에서는 CHIANTI 원자 데이터 (`ChiantiPy`)와 고급 DEM inversion 코드(PINTofALE MCMC, DATA2DEM_REG, XRT_DEM)가 필요하며, 본 노트북은 교육적 목적의 근사임에 유의한다.

## Next steps / 다음 단계
- Install `ChiantiPy` and redo §1 with CHIANTI v9 atomic data for accurate $G(T)$ curves.
- Implement full coronal-model statistical-equilibrium solver for a multi-level ion (e.g., Fe XII 5-level model).
- Apply DEM inversion to real Hinode/EIS or SDO/AIA observations.
- Compare MCMC (PINTofALE) vs Tikhonov (DATA2DEM_REG) on the same data.